# Stable Diffusion VAE embeddings for NSD

Extract deterministic `stabilityai/sd-vae-ft-mse` latents for the NSD 73K stimuli, then optionally compress them with PCA into a smaller RDM-ready feature matrix.

In [ ]:
# Uncomment this cell if the environment is missing the VAE dependencies.
# %pip install -q diffusers transformers accelerate safetensors h5py tqdm scikit-learn

In [1]:
from pathlib import Path
import json
import os
import time

import h5py
import numpy as np
from numpy.lib.format import open_memmap
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

try:
    from diffusers import AutoencoderKL
except ImportError as exc:
    raise ImportError(
        "Missing diffusers. Run the install cell above, then restart the notebook kernel."
    ) from exc

torch.backends.cuda.matmul.allow_tf32 = True

In [2]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "lucas_exploration").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Could not find the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
LUCAS_DIR = PROJECT_ROOT / "lucas_exploration"

# Match the local NSD path used in the other exploration notebooks. Change this if needed.
NSD_DIR = Path("/media/chuddy/120876114737F70A/data/NSD")
STIM_HDF5 = NSD_DIR / "nsddata_stimuli" / "stimuli" / "nsd" / "nsd_stimuli.hdf5"

OUTPUT_DIR = LUCAS_DIR / "results" / "saved_embeddings"
MODEL_ID = "stabilityai/sd-vae-ft-mse"

# Stable Diffusion VAE downsamples by 8. IMAGE_SIZE=256 gives 4 x 32 x 32 = 4096 features.
IMAGE_SIZE = 256
BATCH_SIZE = 64

# Use START/STOP for a tiny smoke test. Leave as 0/None for the full NSD 73K matrix.
START_INDEX = 0
STOP_INDEX = None

RUN_PCA = True
PCA_COMPONENTS = 256
PCA_BATCH_SIZE = 2048

OVERWRITE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"NSD stimulus HDF5: {STIM_HDF5}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")

Project root: /home/chuddy/dev/neuroconnectionism
NSD stimulus HDF5: /media/chuddy/120876114737F70A/data/NSD/nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5
Output dir: /home/chuddy/dev/neuroconnectionism/lucas_exploration/results/saved_embeddings
Device: cuda


In [3]:
def open_nsd_stimulus_dataset(stim_hdf5):
    """Open the NSD image HDF5 and return (file_handle, image_dataset)."""
    if not stim_hdf5.exists():
        raise FileNotFoundError(
            f"Could not find {stim_hdf5}. Point NSD_DIR/STIM_HDF5 at nsd_stimuli.hdf5."
        )

    h5_file = h5py.File(stim_hdf5, "r")
    for key in ("imgBrick", "stimuli", "images"):
        if key in h5_file and isinstance(h5_file[key], h5py.Dataset):
            return h5_file, h5_file[key]

    datasets = []

    def collect_dataset(name, obj):
        if isinstance(obj, h5py.Dataset) and obj.ndim >= 3:
            datasets.append(name)

    h5_file.visititems(collect_dataset)
    if len(datasets) == 1:
        return h5_file, h5_file[datasets[0]]

    h5_file.close()
    raise KeyError(
        "Could not identify the image dataset inside the HDF5. "
        f"Candidate datasets were: {datasets}"
    )


def prepare_image_batch(batch, image_size, device, dtype):
    """Convert uint8 NSD images to VAE input tensor in [-1, 1]."""
    arr = np.asarray(batch)
    x = torch.from_numpy(arr)

    if x.ndim != 4:
        raise ValueError(f"Expected a 4D image batch, got shape {tuple(x.shape)}")

    if x.shape[-1] in (1, 3, 4):
        x = x[..., :3].permute(0, 3, 1, 2)
    elif x.shape[1] in (1, 3, 4):
        x = x[:, :3]
    else:
        raise ValueError(f"Could not infer channel axis for image shape {tuple(x.shape)}")

    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)

    x = x.float()
    if x.max() > 2.0:
        x = x / 255.0

    x = F.interpolate(x, size=(image_size, image_size), mode="bicubic", align_corners=False)
    x = x.clamp(0.0, 1.0)
    x = x * 2.0 - 1.0
    return x.to(device=device, dtype=dtype, non_blocking=True)


def encode_latents(vae, image_batch, image_size, device, dtype):
    x = prepare_image_batch(image_batch, image_size, device, dtype)
    with torch.inference_mode():
        posterior = vae.encode(x).latent_dist
        z = posterior.mode()
        z = z * getattr(vae.config, "scaling_factor", 1.0)
    return z.flatten(1).float().cpu().numpy()

In [4]:
device = torch.device(DEVICE)
vae_dtype = torch.float16 if device.type == "cuda" else torch.float32

print(f"Loading {MODEL_ID}...")
vae = AutoencoderKL.from_pretrained(MODEL_ID, torch_dtype=vae_dtype)
vae = vae.to(device).eval()
print("VAE ready.")

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading stabilityai/sd-vae-ft-mse...


/home/chuddy/dev/neuroconnectionism/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


VAE ready.


In [5]:
stim_file, images = open_nsd_stimulus_dataset(STIM_HDF5)
n_total = int(images.shape[0])
start = int(START_INDEX)
stop = n_total if STOP_INDEX is None else min(int(STOP_INDEX), n_total)

if start < 0 or stop <= start or stop > n_total:
    stim_file.close()
    raise ValueError(f"Invalid START/STOP range: start={start}, stop={stop}, n_total={n_total}")

range_suffix = "" if (start == 0 and stop == n_total) else f"_rows{start}-{stop}"
raw_latents_path = OUTPUT_DIR / f"nsd_sdvae_ft_mse_latents_{IMAGE_SIZE}px{range_suffix}.npy"
raw_meta_path = OUTPUT_DIR / f"nsd_sdvae_ft_mse_latents_{IMAGE_SIZE}px{range_suffix}_meta.json"
partial_latents_path = raw_latents_path.with_name(raw_latents_path.stem + ".partial.npy")

if raw_latents_path.exists() and not OVERWRITE:
    stim_file.close()
    raise FileExistsError(f"{raw_latents_path} already exists. Set OVERWRITE=True to regenerate.")
if partial_latents_path.exists() and not OVERWRITE:
    stim_file.close()
    raise FileExistsError(f"{partial_latents_path} already exists. Delete it or set OVERWRITE=True.")
if partial_latents_path.exists():
    partial_latents_path.unlink()

first_end = min(start + BATCH_SIZE, stop)
first_latents = encode_latents(vae, images[start:first_end], IMAGE_SIZE, device, vae_dtype)
latent_dim = int(first_latents.shape[1])

latents = open_memmap(
    partial_latents_path,
    mode="w+",
    dtype=np.float32,
    shape=(stop - start, latent_dim),
)
latents[: first_end - start] = first_latents
latents.flush()

print(f"Images: {stop - start:,} of {n_total:,}")
print(f"Latent dim: {latent_dim:,}")
print(f"Writing: {partial_latents_path}")

t0 = time.time()
for batch_start in tqdm(range(first_end, stop, BATCH_SIZE), desc="Encoding SD-VAE latents"):
    batch_stop = min(batch_start + BATCH_SIZE, stop)
    batch_latents = encode_latents(vae, images[batch_start:batch_stop], IMAGE_SIZE, device, vae_dtype)
    out_start = batch_start - start
    out_stop = batch_stop - start
    latents[out_start:out_stop] = batch_latents
    latents.flush()

del latents
os.replace(partial_latents_path, raw_latents_path)

metadata = {
    "model_id": MODEL_ID,
    "source_hdf5": str(STIM_HDF5),
    "image_size": IMAGE_SIZE,
    "start_index": start,
    "stop_index": stop,
    "num_images": stop - start,
    "latent_dim": latent_dim,
    "feature": "AutoencoderKL posterior mode, flattened, multiplied by scaling_factor",
    "dtype": "float32",
    "elapsed_seconds": time.time() - t0,
}
raw_meta_path.write_text(json.dumps(metadata, indent=2))
stim_file.close()

print(f"Saved raw latents: {raw_latents_path}")
print(f"Saved metadata: {raw_meta_path}")

/home/chuddy/dev/neuroconnectionism/.venv/lib/python3.12/site-packages/diffusers/models/attention_processor.py:2767: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:383.)
  hidden_states = F.scaled_dot_product_attention(


Images: 73,000 of 73,000
Latent dim: 4,096
Writing: /home/chuddy/dev/neuroconnectionism/lucas_exploration/results/saved_embeddings/nsd_sdvae_ft_mse_latents_256px.partial.npy


Encoding SD-VAE latents:   0%|          | 0/1140 [00:00<?, ?it/s]

Saved raw latents: /home/chuddy/dev/neuroconnectionism/lucas_exploration/results/saved_embeddings/nsd_sdvae_ft_mse_latents_256px.npy
Saved metadata: /home/chuddy/dev/neuroconnectionism/lucas_exploration/results/saved_embeddings/nsd_sdvae_ft_mse_latents_256px_meta.json


In [ ]:
if RUN_PCA:
    from sklearn.decomposition import IncrementalPCA

    raw_latents = np.load(raw_latents_path, mmap_mode="r")
    if raw_latents.shape[0] < PCA_COMPONENTS:
        raise ValueError(
            f"Need at least {PCA_COMPONENTS} rows for PCA, got {raw_latents.shape[0]}. "
            "Use fewer PCA_COMPONENTS or run a larger image range."
        )

    pca_batch_size = max(int(PCA_BATCH_SIZE), int(PCA_COMPONENTS) * 4)
    pca_path = OUTPUT_DIR / f"nsd_sdvae_ft_mse_pca{PCA_COMPONENTS}_{IMAGE_SIZE}px{range_suffix}.npy"
    pca_meta_path = OUTPUT_DIR / f"nsd_sdvae_ft_mse_pca{PCA_COMPONENTS}_{IMAGE_SIZE}px{range_suffix}_meta.json"
    partial_pca_path = pca_path.with_name(pca_path.stem + ".partial.npy")

    if pca_path.exists() and not OVERWRITE:
        raise FileExistsError(f"{pca_path} already exists. Set OVERWRITE=True to regenerate.")
    if partial_pca_path.exists() and not OVERWRITE:
        raise FileExistsError(f"{partial_pca_path} already exists. Delete it or set OVERWRITE=True.")
    if partial_pca_path.exists():
        partial_pca_path.unlink()

    def pca_batch_ranges(n_rows, batch_size, min_batch_size):
        ranges = []
        batch_start = 0
        while batch_start < n_rows:
            batch_stop = min(batch_start + batch_size, n_rows)
            remainder = n_rows - batch_stop
            if 0 < remainder < min_batch_size:
                batch_stop = n_rows
            ranges.append((batch_start, batch_stop))
            batch_start = batch_stop
        return ranges

    pca_ranges = pca_batch_ranges(raw_latents.shape[0], pca_batch_size, PCA_COMPONENTS)
    ipca = IncrementalPCA(n_components=PCA_COMPONENTS, batch_size=pca_batch_size)

    print("Fitting IncrementalPCA...")
    for batch_start, batch_stop in tqdm(pca_ranges, desc="PCA partial_fit"):
        ipca.partial_fit(np.asarray(raw_latents[batch_start:batch_stop], dtype=np.float32))

    pca_features = open_memmap(
        partial_pca_path,
        mode="w+",
        dtype=np.float32,
        shape=(raw_latents.shape[0], PCA_COMPONENTS),
    )

    print("Transforming latents into PCA features...")
    for batch_start, batch_stop in tqdm(pca_ranges, desc="PCA transform"):
        batch = np.asarray(raw_latents[batch_start:batch_stop], dtype=np.float32)
        pca_features[batch_start:batch_stop] = ipca.transform(batch).astype(np.float32)
        pca_features.flush()

    del pca_features
    os.replace(partial_pca_path, pca_path)

    pca_metadata = {
        "source_latents": str(raw_latents_path),
        "num_images": int(raw_latents.shape[0]),
        "input_dim": int(raw_latents.shape[1]),
        "pca_components": int(PCA_COMPONENTS),
        "explained_variance_ratio_sum": float(np.sum(ipca.explained_variance_ratio_)),
        "dtype": "float32",
    }
    pca_meta_path.write_text(json.dumps(pca_metadata, indent=2))

    print(f"Saved PCA features: {pca_path}")
    print(f"Saved PCA metadata: {pca_meta_path}")
else:
    print("RUN_PCA=False, skipping PCA compression.")

In [ ]:
# The RDM helper expects a 73K x n_features array in NSD order.
# For the default full run, this is the model name/path pair to add to get_name2file_dict.py later:
model_name = f"sdvae_ft_mse_pca{PCA_COMPONENTS}"
embedding_path = OUTPUT_DIR / f"nsd_sdvae_ft_mse_pca{PCA_COMPONENTS}_{IMAGE_SIZE}px{range_suffix}.npy"

print("Model name:", model_name)
print("Embedding path:", embedding_path)
print("Shape:", np.load(embedding_path, mmap_mode="r").shape if embedding_path.exists() else "not created")